# MERRA-2 gap fill for the study square, 2010-2024

Companion to `SourceRegion_MAIAC_AOD_Injection_2012_2024_Colab.ipynb`; same
layout, same band naming, same Drive export.

Re-exports the **whole square for every day 2010-2024** for all three
collections, which is simpler than splicing three partial fixes onto the
2012-2024 archives and guarantees one consistent grid.

| gap in the existing data | fixed here |
|---|---|
| south strip 38.5-41.0 N empty | region reaches 38.04 N (grid-aligned) |
| 2010-2011 exists only inside Ontario | `START_YEAR = 2010`, whole square |
| no column mass anywhere | AER carries the five `*CMASS` bands |
| AER lacks surface/extinction outside Ontario | AER carries all 15 bands |

## Grid

Pixels are pinned to the existing archives' lattice, read from
`CAN_SRC_MERRA_AER_2013.tif`:

```
pixel  = 0.4940734062657368 deg
origin = (-136.8583335356091, 69.66435028346889)   # upper-left corner
```

Passed as `crs='EPSG:4326'` + an explicit `crsTransform` list, so the new
pixels land on the same lattice indices as the old ones. (Not `scale=55000`:
that only approximates 0.494073 deg and drifts from the reference grid.)

The region is snapped to whole pixel boundaries — column 76..144, row 20..64 —
so no edge pixel is partially covered and masked.

## Products

| PRODUCT | asset | bands/day |
|---|---|---|
| `AER` | `NASA/GSFC/MERRA/aer/2` | 15 |
| `FLX` | `NASA/GSFC/MERRA/flx/2` | 7 |
| `SLV` | `NASA/GSFC/MERRA/slv/2` | 12 |

AER bands: `BCSMASS, OCSMASS, SO4SMASS, SSSMASS25, DUSMASS25` (surface),
`BCCMASS, OCCMASS, SO4CMASS, SSCMASS25, DUCMASS25` (column),
`BCEXTTAU, OCEXTTAU, SUEXTTAU, DUEXTTAU, TOTEXTTAU` (extinction).

Each day is the mean of that UTC day's hourly images, as in the existing
archives. Empty days are written as **-999**.

**The daily window is UTC-6.** Earth Engine's MERRA-2 collections are
*hourly* (cadence 1 h, stamped HH:30), so `filterDate(D 06:00, D+1 06:00).mean()`
is exactly the UTC-6 day the PM2.5 target uses. The 2012-2024 archives were built
with the 00:00-24:00 UTC window, which is why they sit 6 h off the target.


In [ ]:
!pip install -q earthengine-api geemap

In [ ]:
import calendar
import time
import ee

PROJECT_ID = 'my-project-pm25predictionon'  # change this

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print('Earth Engine ready')

In [ ]:
START_YEAR = 2010
END_YEAR = 2024
DRIVE_FOLDER = 'SourceRegion_MERRA_2010_2024_v2'

# ---- reference grid, read from the existing Canada archives -------------
# DO NOT change these unless you also change the existing files; everything
# downstream assumes the new pixels sit on this lattice.
PX = 0.4940734062657368          # pixel size, degrees
LON0 = -136.8583335356091        # upper-left corner longitude
LAT0 = 69.66435028346889         # upper-left corner latitude

# Pixel index window: whole pixels only, so no edge pixel is half-masked.
# col 76..144 covers -99.31 .. -65.71 ; row 20..64 covers 59.78 .. 38.04
K0, K1 = 76, 144
M0, M1 = 20, 64

REGION_BBOX = [
    LON0 + K0 * PX,              # west  -99.3088
    LAT0 - M1 * PX,              # south  38.0437
    LON0 + K1 * PX,              # east  -65.7118
    LAT0 - M0 * PX,              # north  59.7829
]
ROI = ee.Geometry.Rectangle(REGION_BBOX, geodesic=False)

CRS = 'EPSG:4326'
CRS_TRANSFORM = [PX, 0, LON0, 0, -PX, LAT0]   # [xscale, xskew, xorigin, yskew, yscale, yorigin]

# MERRA-2 images are HOURLY (GEE cadence 1 h, stamped HH:30). The daily
# mean is therefore a window we choose:
#   'UTC'   -> 00:00-24:00 UTC  (what the 2012-2024 archives used)
#   'UTC-6' -> 06:00-06:00 UTC  (the PM2.5 target day; 24 hourly images)
DAY_MODE = 'UTC-6'

MAX_PIXELS = 1_000_000_000_000
SUBMIT_SLEEP = 0.3
PRODUCTS = ['AER', 'FLX', 'SLV']

ASSETS = {
    'AER': 'NASA/GSFC/MERRA/aer/2',
    'FLX': 'NASA/GSFC/MERRA/flx/2',
    'SLV': 'NASA/GSFC/MERRA/slv/2',
}

# '25' is the PM2.5 size fraction; BC, OC and SO4 are fine-mode only and carry
# no suffix. Organic carbon column mass is OCCMASS (OCC + MASS).
VARIABLES = {
    'AER': [
        'BCSMASS', 'OCSMASS', 'SO4SMASS', 'SSSMASS25', 'DUSMASS25',      # surface
        'BCCMASS', 'OCCMASS', 'SO4CMASS', 'SSCMASS25', 'DUCMASS25',       # column
        'BCEXTTAU', 'OCEXTTAU', 'SUEXTTAU', 'DUEXTTAU', 'TOTEXTTAU',     # extinction
    ],
    'FLX': ['PBLH', 'QLML', 'TLML', 'ULML', 'VLML', 'SPEED', 'SPEEDMAX'],
    'SLV': ['U2M', 'V2M', 'U10M', 'V10M', 'U50M', 'V50M',
            'U250', 'V250', 'U500', 'V500', 'U850', 'V850'],
}

print('region  :', [round(v, 4) for v in REGION_BBOX])
print('pixels  :', K1 - K0, 'x', M1 - M0)
print('transform:', CRS_TRANSFORM)

# ---- fail early if any band name is wrong -------------------------------
for _p in PRODUCTS:
    _bands = ee.ImageCollection(ASSETS[_p]).filterBounds(ROI).first().bandNames().getInfo()
    _missing = [v for v in VARIABLES[_p] if v not in _bands]
    assert not _missing, f'{_p}: bands not found in {ASSETS[_p]}: {_missing}'
    print(f'{_p:3s} {len(VARIABLES[_p]):2d} bands verified')

# show any asset bands we are NOT exporting, so nothing is missed silently
for _p in PRODUCTS:
    _bands = ee.ImageCollection(ASSETS[_p]).filterBounds(ROI).first().bandNames().getInfo()
    _extra = [b for b in _bands if b not in VARIABLES[_p]]
    if _extra:
        print(f'    {_p} also has (not exported): {_extra}')

In [ ]:
def merra_daily_image(date: ee.Date, asset: str, variables: list) -> ee.Image:
    start = ee.Date(date)
    if DAY_MODE == 'UTC-6':
        start = start.advance(6, 'hour')
    end = start.advance(1, 'day')
    coll = (
        ee.ImageCollection(asset)
        .filterDate(start, end)
        .filterBounds(ROI)
        .select(variables)
    )
    reduced = coll.mean()
    return ee.Image(ee.Algorithms.If(coll.size().gt(0), reduced,
                                     empty_merra_image(start, variables))).set(
        'system:time_start', start.millis()
    )


def empty_merra_image(date: ee.Date, variables: list) -> ee.Image:
    return ee.Image.constant([-999] * len(variables)).toFloat().rename(variables).set(
        'system:time_start', date.millis()
    )


def merra_month_stack(year: int, month: int, asset: str, variables: list) -> ee.Image:
    start = ee.Date.fromYMD(year, month, 1)
    n_days = calendar.monthrange(year, month)[1]
    stack = None
    for i in range(n_days):
        d = start.advance(i, 'day')
        tag = d.format('YYYYMMdd')
        daily = merra_daily_image(d, asset, variables)
        renamed = daily.rename([tag.cat('_' + v) for v in variables])
        stack = renamed if stack is None else stack.addBands(renamed)
    return stack.toFloat()


def export_month(product: str, year: int, month: int) -> None:
    tag = f'{year:04d}_{month:02d}'
    task = ee.batch.Export.image.toDrive(
        image=merra_month_stack(year, month, ASSETS[product], VARIABLES[product]),
        description=f'SRC_MERRA_{product}_{tag}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'source_region_merra_{product.lower()}_daily_stack_{tag}',
        region=ROI,
        crs=CRS,
        crsTransform=CRS_TRANSFORM,
        maxPixels=MAX_PIXELS,
        fileFormat='GeoTIFF',
    )
    task.start()
    print('started', product, tag)

In [ ]:
# ---- smoke test ---------------------------------------------------------
# One month, one product. Expect 31 days x 15 bands = 465 bands, named
# 20100101_BCSMASS ... 20100131_TOTEXTTAU, on a 68 x 44 pixel grid.

export_month('AER', 2010, 1)
print('check the task, then verify the GeoTIFF before running the next cell')

In [ ]:
# ---- submit everything --------------------------------------------------
# 3 products x 15 years x 12 months = 540 tasks, ~3 min to submit.
# Only run this once the smoke test looks right.

for product in PRODUCTS:
    for year in range(START_YEAR, END_YEAR + 1):
        for month in range(1, 13):
            export_month(product, year, month)
            time.sleep(SUBMIT_SLEEP)
    print('submitted', product)

print('all submitted')